# Season outputs loader
Helper cells to flexibly load season run outputs stored under `data/season_outputs/<run_id>`.
- Lists available runs
- Loads trip log, day summary, season person snapshots, SP day summary
- Discovers all `day_*_model_ts.parquet` files into a dict keyed by day index

Update `RUN_ID` below to point at the run you want to analyze.

In [4]:
import os
from pathlib import Path
import subprocess

# Get the top-level directory of the current git repo
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)

os.chdir(PROJECT_ROOT)



%pwd

import importlib

import season.analysis_helpers as ah

# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
ah.list_runs()

['example_season_001',
 'light_season_001',
 'speed_test',
 'speed_test2',
 'speed_test3',
 'speed_test4',
 'speed_test5',
 'speed_test6',
 'store_data_2days_test',
 'test',
 'test2k',
 'two_week_example_001',
 'volume_toll_1',
 'volume_toll_2',
 'volume_toll_3',
 'volume_toll_4',
 'volume_toll_5',
 'volume_toll_6']

## Load a run
Set `RUN_ID` to one of the `available_runs` above.

In [6]:
RUN_ID = 'speed_test6' 

run_data = ah.load_run(RUN_ID)
run_data_keys = {k: (list(v.keys()) if k == 'model_ts' else (None if v is None else getattr(v, 'shape', None))) for k, v in run_data.items()}
run_data_keys


{'run_dir': None,
 'trip_log': (4500, 9),
 'day_summary': (3, 20),
 'season_person_log': (4500, 18),
 'sp_day_summary': (3, 11),
 'model_ts': [0, 1, 2],
 'season_summary': None}

## Quick peeks
Uncomment and run the snippets you need once a run is loaded.

In [10]:
print('='*20)
print('Model time series day 0')
print('Tier 1 data collected at 60s intervals')
print('='*20)
model_ts = run_data['model_ts']
display(model_ts[0].head()) if model_ts is not None and len(model_ts) > 0 else print('No model time series data')


Model time series day 0
Tier 1 data collected at 60s intervals


,step,p_generate,current_toll,vehicle_count,active_cars,bus_riders_waiting,active_buses,total_finished,bus_mode_share_recent,recent_travel_time_avg,rolling_count_vehicles_generated,rolling_count_persons_generated,implicit_sl_delta_bin_0,implicit_sl_delta_bin_1,implicit_sl_delta_bin_2,implicit_sl_delta_bin_3,implicit_sl_delta_bin_4
0,60,0.176464,0.0,6,6,0,0,0,0.0,NaN,0,0,0,0,0,1,5
1,120,0.178048,0.0,13,13,0,0,0,0.0,NaN,7,7,0,0,0,3,10
2,180,0.179631,0.0,26,26,0,0,0,0.0,NaN,20,20,0,0,2,7,17
3,240,0.181214,0.0,36,36,0,0,0,0.0,NaN,30,30,0,0,2,9,25
4,300,0.182798,0.0,44,44,0,0,0,0.0,NaN,38,38,0,0,0,17,27


In [3]:
print('='*100)
print('Season Summary - one row per SEASON, with aggregate metrics')
print('='*100)
run_data['season_summary']

print('='*100)
print('Day summary - one row per DAY, with aggregate metrics')
print('='*100)
day_summary = run_data['day_summary']
display(day_summary.head()) if day_summary is not None else print('No day summary data')  

print('='*100)
print('Trip log - one row per PERSON per DAY')
print('='*100)
trip_log = run_data['trip_log']
display(trip_log.head()) if trip_log is not None else print('No trip log data')  

print('='*100)
print('Season person log - one row per PERSON per DAY, with their mode and travel times')
print('='*100)
season_person_log = run_data['season_person_log']
display(season_person_log.loc[season_person_log.person_id == 1])



Season Summary - one row per SEASON, with aggregate metrics


NameError: name 'run_data' is not defined

In [ ]:
from season.analysis_helpers import plot_model_ts_interactive
plot_model_ts_interactive(model_ts, run_id=RUN_ID)


In [ ]:
from season.analysis_helpers import plot_realized_cost_means_with_total
plot_realized_cost_means_with_total(trip_log)
from season.analysis_helpers import plot_realized_cost_boxplots
plot_realized_cost_boxplots(trip_log)
